# AlexNet — ImageNet subset

Dikonversi dari `02_alexnet_imagenet.py`. Jalankan sel secara berurutan dengan kernel Python yang memiliki `torch` dan `torchvision`. Sel terakhir memulai pelatihan.

Dataset Tiny ImageNet berada di `data/tiny-imagenet-200/` dalam folder `notebooks`. Loader mendukung direktori kerja root proyek maupun `notebooks`, membaca kelas train dari subfoldernya, dan membaca label validation dari `val/val_annotations.txt`. Sesuaikan `IMAGENET_ROOT` jika dataset dipindahkan.


## Import dan konfigurasi


In [13]:
import random
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms

SEED = 42
BATCH_SIZE = 32
EPOCHS = 5
TRAIN_SAMPLES = 4000
VAL_SAMPLES = 800
IMAGENET_ROOT = "./data/tiny-imagenet-200"



## Seed untuk reproduksibilitas


In [14]:
def set_seed(seed=SEED):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)



## Dataset dan DataLoader


In [15]:
class TinyImageNetValidation(Dataset):
    def __init__(self, root, class_to_idx, transform=None):
        self.root = Path(root)
        self.class_to_idx = dict(class_to_idx)
        self.classes = sorted(class_to_idx, key=class_to_idx.get)
        self.transform = transform
        self.samples = []
        annotations = self.root / "val_annotations.txt"
        for line in annotations.read_text().splitlines():
            filename, class_id, *_ = line.split()
            self.samples.append((self.root / "images" / filename, self.class_to_idx[class_id]))
        self.targets = [label for _, label in self.samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        image = datasets.folder.default_loader(str(image_path))
        if self.transform is not None:
            image = self.transform(image)
        return image, label


def get_dataloaders():
    dataset_root = Path(IMAGENET_ROOT).expanduser()
    if not dataset_root.is_absolute() and not dataset_root.is_dir():
        notebook_root = Path("notebooks") / dataset_root
        if notebook_root.is_dir():
            dataset_root = notebook_root
    dataset_root = dataset_root.resolve()
    train_dir = dataset_root / "train"
    val_dir = dataset_root / "val"

    if not train_dir.is_dir() or not val_dir.is_dir():
        raise FileNotFoundError(
            f"Dataset tidak ditemukan di {dataset_root}. "
            "Sesuaikan IMAGENET_ROOT ke folder tiny-imagenet-200 yang berisi train/ dan val/."
        )

    transform_train = transforms.Compose([
        transforms.Resize((144, 144)),
        transforms.RandomCrop(128),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406),
                             (0.229, 0.224, 0.225))
    ])

    transform_val = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406),
                             (0.229, 0.224, 0.225))
    ])

    train_set = datasets.ImageFolder(train_dir, transform=transform_train)
    if (val_dir / "images").is_dir():
        val_set = TinyImageNetValidation(val_dir, train_set.class_to_idx, transform=transform_val)
    else:
        val_set = datasets.ImageFolder(val_dir, transform=transform_val)

    if train_set.classes != val_set.classes:
        raise ValueError("Class folder pada train dan val harus sama.")

    g = torch.Generator().manual_seed(SEED)
    train_n = min(TRAIN_SAMPLES, len(train_set))
    val_n = min(VAL_SAMPLES, len(val_set))

    train_idx = torch.randperm(len(train_set), generator=g)[:train_n]
    val_idx = torch.randperm(len(val_set), generator=g)[:val_n]

    train_loader = DataLoader(
        Subset(train_set, train_idx),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    val_loader = DataLoader(
        Subset(val_set, val_idx),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    return train_loader, val_loader, len(train_set.classes)



In [16]:
train_loader, val_loader, num_classes = get_dataloaders()

# Mengambil batch pertama
images, labels = next(iter(train_loader))

print(labels)
print(labels.size())   # Bentuk tensor label
print(labels.size(0))  # Jumlah gambar dalam batch


tensor([ 87, 114,   1,  17,  75, 137,   1, 109,  37, 165,  61, 151,  84,  79,
        187,  31, 162,  30,  36, 161,  21,  83,  30,  69,  12,  82, 177,  22,
        179,  70, 107,  23])
torch.Size([32])
32


## Fungsi pelatihan dan evaluasi


In [17]:
def train_model(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

def run_training(model, model_name):
    set_seed()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_loader, val_loader, num_classes = get_dataloaders()

    model = model(num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    history = {
        "train_loss": [], "val_loss": [],
        "train_acc": [], "val_acc": []
    }

    print(f"Model: {model_name}")
    print("Dataset: ImageNet subset")
    print(f"Classes: {num_classes}")
    print(f"Device: {device}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_model(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

    return model, history, val_loader, device




## Arsitektur model


In [ ]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2),
            nn.Conv2d(32, 64, 5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 96, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((3, 3))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(96 * 3 * 3, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))



## Jalankan pelatihan


In [19]:
trained_model, history, val_loader, device = run_training(AlexNet, "AlexNet")


Model: AlexNet
Dataset: ImageNet subset
Classes: 200
Device: cuda
Parameters: 933,608
Epoch 01/5 | train_loss=5.3000 | train_acc=0.0055 | val_loss=5.3020 | val_acc=0.0075
Epoch 02/5 | train_loss=5.2853 | train_acc=0.0083 | val_loss=5.3991 | val_acc=0.0088
Epoch 03/5 | train_loss=5.2459 | train_acc=0.0090 | val_loss=5.2730 | val_acc=0.0163
Epoch 04/5 | train_loss=5.1784 | train_acc=0.0112 | val_loss=5.2370 | val_acc=0.0075
Epoch 05/5 | train_loss=5.1405 | train_acc=0.0105 | val_loss=5.2102 | val_acc=0.0075
